# Time Series Analysis Study Guide

Hands-on time series guide using pandas, numpy, matplotlib, and statsmodels.

## 1. 1. DateTime Indexing

Pandas DatetimeIndex enables powerful time-based selection, alignment, and operations. Use pd.to_datetime() to parse dates and set_index() to create a time series.

**Creating a DatetimeIndex**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
dates = pd.date_range('2024-01-01', periods=10, freq='D')
s = pd.Series(rng.integers(100, 500, 10), index=dates)
print(s)
print('\nIndex type:', type(s.index))
print('Freq:', s.index.freq)
print('First date:', s.index[0])
print('Last date:', s.index[-1])

**Parsing and indexing with pd.to_datetime**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(7)
df = pd.DataFrame({
    'date':    ['2024-01-15', '2024-02-20', '2024-03-05', '2024-04-10'],
    'revenue': rng.integers(1000, 9999, 4),
    'units':   rng.integers(10, 200, 4),
})
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')
print(df)
print('\n.dt accessor on a column:')
df2 = df.reset_index()
df2['year']  = df2['date'].dt.year
df2['month'] = df2['date'].dt.month
df2['dow']   = df2['date'].dt.day_name()
print(df2[['date','year','month','dow']])

**Time-based slicing and selection**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(0)
idx = pd.date_range('2023-01-01', '2024-12-31', freq='D')
ts = pd.Series(rng.normal(100, 15, len(idx)), index=idx)

# Slice by string
print('Jan 2024:', ts['2024-01'].shape)
print('Q1 2024:', ts['2024-01':'2024-03'].shape)
print('A specific week:', ts['2024-06-10':'2024-06-16'].values.round(2))

# Boolean mask
print('\nDays with value > 130:', (ts > 130).sum())
print('Weekend values mean:', ts[ts.index.dayofweek >= 5].mean().round(2))

**Time zones and date arithmetic**

In [ ]:
import pandas as pd
import numpy as np

# Create UTC series and convert timezone
idx = pd.date_range('2024-01-01 00:00', periods=6, freq='6h', tz='UTC')
ts = pd.Series(range(6), index=idx)
print('UTC:')
print(ts)

# Convert to US/Eastern
ts_eastern = ts.tz_convert('US/Eastern')
print('\nUS/Eastern:')
print(ts_eastern)

# Date arithmetic
today = pd.Timestamp('2024-06-15')
print('\n30 days later:', today + pd.Timedelta(days=30))
print('Next Monday:', today + pd.offsets.Week(weekday=0))
print('End of month:', today + pd.offsets.MonthEnd(0))

# Period vs Timestamp
period = pd.Period('2024-Q2', freq='Q')
print('\nQ2 2024 start:', period.start_time)
print('Q2 2024 end:  ', period.end_time)

### Real-World Scenario

> An e-commerce analyst needs to parse mixed-format sales timestamps, localize to UTC, and create a clean DatetimeIndex for downstream aggregation.

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
raw = pd.DataFrame({
    'timestamp': ['2024-01-15 09:32', '2024/02/20 14:05', 'March 5, 2024 11:00', '2024-04-10T08:45:00'],
    'store':     ['NYC', 'LA', 'CHI', 'HOU'],
    'amount':    rng.uniform(50, 500, 4).round(2),
})

# Parse mixed formats
raw['ts_parsed'] = pd.to_datetime(raw['timestamp'], infer_datetime_format=True)
raw = raw.set_index('ts_parsed').sort_index()
raw.index = raw.index.tz_localize('US/Eastern').tz_convert('UTC')

print('Cleaned time series:')
print(raw[['store', 'amount']])
print('\nIndex timezone:', raw.index.tz)
print('Date range:', raw.index[0], 'to', raw.index[-1])

### Practice: Custom DatetimeIndex

Create a DatetimeIndex for every Monday in 2024. Build a Series with random weekly sales (uniform 1000-5000). Print the total sales per quarter using .resample('QE').sum().

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(0)
# TODO: create weekly DatetimeIndex (Mondays only) for 2024
# Hint: pd.date_range(..., freq='W-MON')

# TODO: create Series with random sales

# TODO: resample to quarterly and print


## 2. 2. Resampling & Frequency Conversion

Resampling changes the time frequency of a series — downsampling aggregates to lower frequency, upsampling interpolates to higher frequency.

**Downsampling with resample()**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', periods=365, freq='D')
daily = pd.Series(rng.uniform(100, 500, 365), index=idx, name='sales')

# Downsample to different frequencies
print('Weekly sum (first 4):')
print(daily.resample('W').sum().head(4).round(2))

print('\nMonthly stats:')
monthly = daily.resample('ME').agg(['sum','mean','max','min'])
print(monthly.round(2))

print('\nQuarterly mean:')
print(daily.resample('QE').mean().round(2))

**OHLC resampling for financial data**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(0)
idx = pd.date_range('2024-01-01', periods=365, freq='D')
price = 100 + np.cumsum(rng.normal(0, 1.5, 365))
ts = pd.Series(price, index=idx, name='price')

# OHLC aggregation
weekly_ohlc = ts.resample('W').ohlc()
print('Weekly OHLC (first 5):')
print(weekly_ohlc.head().round(2))

# Custom aggregation
monthly_custom = ts.resample('ME').agg(
    open  = ('first'),
    close = ('last'),
    high  = ('max'),
    low   = ('min'),
    range = (lambda x: x.max() - x.min()),
)
print('\nMonthly custom agg:')
print(monthly_custom.round(2))

**Upsampling and interpolation**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(7)
# Monthly data → daily via upsampling
monthly_idx = pd.date_range('2024-01-01', periods=6, freq='ME')
monthly = pd.Series([120, 135, 128, 142, 156, 149], index=monthly_idx, name='revenue')

# Forward fill
daily_ffill = monthly.resample('D').ffill()
# Linear interpolation
daily_interp = monthly.resample('D').interpolate('linear')
# Cubic interpolation
daily_cubic = monthly.resample('D').interpolate('cubic')

print('Monthly original:')
print(monthly)
print('\nDaily (first 10, forward fill):')
print(daily_ffill.head(10).round(2))
print('\nDaily (first 10, linear interp):')
print(daily_interp.head(10).round(2))

**asfreq() vs resample() and timezone-aware resampling**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', periods=24*7, freq='h', tz='UTC')
hourly = pd.Series(rng.uniform(10, 100, len(idx)), index=idx)

# asfreq — select existing values at new frequency (no aggregation)
every_6h = hourly.asfreq('6h')
print('asfreq every 6h (first 8):')
print(every_6h.head(8).round(2))

# Resample by time-of-day
hourly_mean = hourly.groupby(hourly.index.hour).mean()
print('\nMean by hour of day:')
for h, v in hourly_mean.items():
    bar = '█' * int(v / 10)
    print(f'  {h:02d}h: {v:5.1f} {bar}')

### Real-World Scenario

> A retail analyst has daily transaction data for 3 years and needs to compute weekly, monthly, and quarterly revenue summaries with growth rates for an executive report.

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2022-01-01', '2024-12-31', freq='D')
noise = rng.normal(0, 20, len(idx))
trend = np.linspace(500, 800, len(idx))
seasonal = 80 * np.sin(2 * np.pi * np.arange(len(idx)) / 365)
daily = pd.Series(trend + seasonal + noise, index=idx, name='revenue').clip(0)

# Aggregate
weekly  = daily.resample('W').sum()
monthly = daily.resample('ME').sum()
quarterly = daily.resample('QE').sum()

# Growth rates
monthly_growth = monthly.pct_change() * 100
quarterly_growth = quarterly.pct_change() * 100

print('Monthly revenue with MoM growth (last 6):')
report = pd.DataFrame({'revenue': monthly, 'mom_growth_%': monthly_growth}).tail(6)
print(report.round(2))
print('\nQuarterly totals with QoQ growth:')
print(pd.DataFrame({'revenue': quarterly, 'qoq_%': quarterly_growth}).round(2))

### Practice: Resample and Compare

Generate hourly temperature data for January 2024 (mean=5°C, std=8°C, with a sine wave daily cycle). Resample to daily min/mean/max. Find the coldest and warmest day.

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', '2024-01-31 23:00', freq='h')
# TODO: create hourly temp with sine daily cycle + noise

# TODO: resample to daily min/mean/max

# TODO: print coldest and warmest day


## 3. 3. Rolling & Expanding Windows

Rolling windows compute statistics over a sliding fixed-length window. Expanding windows grow from the start of the series. Both are essential for smoothing and feature engineering.

**Rolling mean and standard deviation**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', periods=30, freq='D')
ts = pd.Series(rng.normal(100, 15, 30), index=idx, name='value')

rm7  = ts.rolling(7).mean()
rm14 = ts.rolling(14).mean()
std7 = ts.rolling(7).std()

result = pd.DataFrame({'raw': ts, 'ma7': rm7, 'ma14': rm14, 'std7': std7})
print(result.tail(10).round(2))
print('\nNaN count from rolling(7):', rm7.isna().sum())

**Rolling min_periods and center**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(0)
ts = pd.Series(rng.normal(50, 10, 20), name='signal')

# min_periods: require at least 3 valid values
r_minp = ts.rolling(7, min_periods=3).mean()
# center=True: window centered on current observation
r_center = ts.rolling(7, center=True).mean()

print('Standard rolling(7): first 7 values')
print(ts.rolling(7).mean().head(7).round(2).tolist())
print('\nWith min_periods=3: first 7 values')
print(r_minp.head(7).round(2).tolist())
print('\nCentered rolling(7): first 7 values')
print(r_center.head(7).round(2).tolist())

# Rolling correlation between two series
s2 = pd.Series(rng.normal(50, 10, 20))
print('\nRolling correlation (last 5):')
print(ts.rolling(5).corr(s2).tail().round(3).tolist())

**Expanding windows and cumulative stats**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(7)
ts = pd.Series(rng.exponential(100, 20), name='revenue')

exp_mean = ts.expanding().mean()
exp_std  = ts.expanding().std()
exp_max  = ts.expanding().max()
cumsum   = ts.cumsum()

result = pd.DataFrame({
    'revenue':    ts.round(2),
    'cum_mean':   exp_mean.round(2),
    'cum_std':    exp_std.round(2),
    'cum_max':    exp_max.round(2),
    'cumsum':     cumsum.round(2),
})
print(result)

**Exponentially weighted moving average (EWMA)**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', periods=30, freq='D')
ts = pd.Series(rng.normal(100, 20, 30) + np.linspace(0, 30, 30), index=idx)

# EWM with different spans
ewm5  = ts.ewm(span=5,  adjust=False).mean()
ewm14 = ts.ewm(span=14, adjust=False).mean()
# EWM with alpha (decay factor directly)
ewm_a = ts.ewm(alpha=0.3, adjust=False).mean()

result = pd.DataFrame({'raw': ts, 'ewm5': ewm5, 'ewm14': ewm14, 'ewm_a0.3': ewm_a})
print(result.round(2))
print('\nEWM vs SMA final values:')
print(f'  EWM(span=5):  {ewm5.iloc[-1]:.2f}')
print(f'  SMA(window=5):{ts.rolling(5).mean().iloc[-1]:.2f}')

### Real-World Scenario

> A supply chain analyst needs to flag anomalous daily order volumes by computing a 14-day rolling mean and standard deviation, then marking any day beyond 2 standard deviations as an outlier.

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', periods=90, freq='D')
orders = pd.Series(rng.normal(500, 60, 90), index=idx, name='orders')
# Inject anomalies
orders.iloc[15] = 950
orders.iloc[45] = 120
orders.iloc[72] = 880

roll = orders.rolling(14)
rm   = roll.mean()
rs   = roll.std()
upper = rm + 2 * rs
lower = rm - 2 * rs

anomalies = orders[(orders > upper) | (orders < lower)]
print(f'Total days: {len(orders)}, Anomalies detected: {len(anomalies)}')
print('\nAnomalous days:')
for date, val in anomalies.items():
    print(f'  {date.date()}: {val:.0f} orders (mean={rm[date]:.0f}, '
          f'bounds=[{lower[date]:.0f}, {upper[date]:.0f}])')

### Practice: Bollinger Bands

Generate 60 days of synthetic stock price data starting at $100 with daily returns from N(0.001, 0.02). Compute 20-day SMA and 2-std Bollinger Bands. Print the last 10 rows showing price, upper band, lower band, and whether the price is outside the bands.

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', periods=60, freq='B')
# TODO: create price series with cumulative returns

# TODO: compute 20-day SMA and Bollinger Bands

# TODO: flag days outside bands and print last 10 rows


## 4. 4. Time Series Visualization

Visualizing time series reveals trends, seasonality, and anomalies. Key plots include line charts, seasonal decomposition, ACF/PACF correlograms, and lag plots.

**Basic time series plot with annotations**

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2023-01-01', periods=365, freq='D')
ts = pd.Series(
    100 + np.linspace(0, 50, 365) + 20 * np.sin(2*np.pi*np.arange(365)/365) + rng.normal(0, 5, 365),
    index=idx, name='sales'
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts.index, ts.values, lw=1, color='steelblue', label='Daily')
ax.plot(ts.rolling(30).mean().index, ts.rolling(30).mean(), lw=2, color='tomato', label='30-day MA')
ax.axvline(pd.Timestamp('2023-06-21'), color='green', ls='--', lw=1.5, label='Summer Solstice')
ax.fill_between(ts.index, ts.rolling(30).mean() - ts.rolling(30).std(),
                ts.rolling(30).mean() + ts.rolling(30).std(), alpha=0.2, color='tomato')
ax.set(title='Sales Time Series with 30-Day Moving Average', xlabel='Date', ylabel='Sales')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ts_basic.png', dpi=100)
plt.close()
print('Saved ts_basic.png')

**Seasonal subseries plot**

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

rng = np.random.default_rng(0)
idx = pd.date_range('2022-01-01', periods=36, freq='ME')
monthly = pd.Series(
    200 + 50*np.sin(2*np.pi*np.arange(36)/12) + rng.normal(0, 10, 36),
    index=idx, name='revenue'
)

fig, axes = plt.subplots(3, 4, figsize=(14, 8), sharey=True)
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for m in range(1, 13):
    ax = axes[(m-1)//4][(m-1)%4]
    data = monthly[monthly.index.month == m]
    ax.bar(range(len(data)), data.values, color='steelblue', alpha=0.7)
    ax.axhline(data.mean(), color='tomato', lw=2)
    ax.set_title(month_names[m-1], fontsize=10)
    ax.set_xticks([])
plt.suptitle('Seasonal Subseries Plot — Monthly Revenue', fontsize=13)
plt.tight_layout()
plt.savefig('ts_seasonal_sub.png', dpi=100)
plt.close()
print('Saved ts_seasonal_sub.png')

**ACF and PACF correlograms**

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

try:
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    rng = np.random.default_rng(42)
    # AR(2) process: y_t = 0.7*y_{t-1} - 0.3*y_{t-2} + noise
    n = 200
    y = np.zeros(n)
    eps = rng.normal(0, 1, n)
    for t in range(2, n):
        y[t] = 0.7*y[t-1] - 0.3*y[t-2] + eps[t]
    ts = pd.Series(y)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_acf(ts, lags=20, ax=axes[0], title='ACF — AR(2) Process')
    plot_pacf(ts, lags=20, ax=axes[1], title='PACF — AR(2) Process', method='ywm')
    plt.tight_layout()
    plt.savefig('ts_acf.png', dpi=100)
    plt.close()
    print('Saved ts_acf.png')
    print('ACF at lag 1:', round(ts.autocorr(1), 3))
    print('ACF at lag 2:', round(ts.autocorr(2), 3))
except ImportError:
    print('statsmodels not installed — pip install statsmodels')
    import pandas as pd, numpy as np
    ts = pd.Series(np.random.randn(100))
    print('Manual ACF at lags 1-5:')
    for lag in range(1, 6):
        print(f'  lag {lag}: {ts.autocorr(lag):.3f}')

**Lag plot and return distribution**

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', periods=100, freq='B')
price = 100 + np.cumsum(rng.normal(0, 1.5, 100))
ts = pd.Series(price, index=idx)
returns = ts.pct_change().dropna() * 100

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
# Lag plot
axes[0].scatter(ts.values[:-1], ts.values[1:], alpha=0.5, s=20, color='steelblue')
axes[0].set(title='Lag Plot (lag=1)', xlabel='Price(t)', ylabel='Price(t+1)')

# Return distribution
axes[1].hist(returns, bins=20, color='coral', edgecolor='white', alpha=0.8)
axes[1].set(title='Daily Return Distribution', xlabel='Return (%)', ylabel='Count')

# Rolling volatility
vol = returns.rolling(10).std()
axes[2].plot(vol.index, vol, color='purple', lw=1.5)
axes[2].set(title='10-Day Rolling Volatility', xlabel='Date', ylabel='Std Dev (%)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('ts_lag_dist.png', dpi=100)
plt.close()
print('Saved ts_lag_dist.png')
print(f'Return stats: mean={returns.mean():.2f}%, std={returns.std():.2f}%, skew={returns.skew():.2f}')

### Real-World Scenario

> A business analyst needs to visualize 2 years of weekly e-commerce sales showing trend, seasonality, rolling average, and year-over-year comparison in a single dashboard.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2023-01-02', periods=104, freq='W-MON')
trend = np.linspace(5000, 8000, 104)
seasonal = 1500 * np.sin(2*np.pi*np.arange(104)/52)
sales = pd.Series(trend + seasonal + rng.normal(0, 200, 104), index=idx, name='sales')

y2023 = sales['2023'].values
y2024 = sales['2024'].values[:len(y2023)]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(sales.index, sales, alpha=0.4, lw=1, color='steelblue', label='Weekly')
axes[0].plot(sales.rolling(8).mean().index, sales.rolling(8).mean(), lw=2, color='tomato', label='8-wk MA')
axes[0].set(title='Weekly E-Commerce Sales (2023-2024)', ylabel='Sales ($)')
axes[0].legend(); axes[0].grid(alpha=0.3)

weeks = range(1, min(len(y2023), len(y2024))+1)
axes[1].plot(weeks, y2023[:len(weeks)], label='2023', color='steelblue', lw=2)
axes[1].plot(weeks, y2024[:len(weeks)], label='2024', color='tomato', lw=2)
axes[1].set(title='Year-over-Year Comparison', xlabel='Week', ylabel='Sales ($)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('ts_dashboard.png', dpi=100)
plt.close()
print('Saved ts_dashboard.png')
yoy_growth = (y2024.mean() / y2023.mean() - 1) * 100
print(f'YoY growth: {yoy_growth:+.1f}%')

### Practice: Multi-Series Time Plot

Generate 52 weeks of data for three products (A, B, C) with different trends and seasonalities. Plot all three on the same axis with a legend. Add a 4-week rolling average for each. Save as 'practice_ts.png'.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', periods=52, freq='W')
# TODO: create 3 product series with different patterns

# TODO: plot all 3 with rolling averages

plt.savefig('practice_ts.png', dpi=100)
plt.close()
print('Saved practice_ts.png')

## 5. 5. Trend & Seasonality Decomposition

Decomposition separates a time series into trend, seasonal, and residual components. Additive decomposition (Y = T + S + R) works for constant amplitude seasonality; multiplicative (Y = T × S × R) for growing amplitude.

**Classical decomposition with statsmodels**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    rng = np.random.default_rng(42)
    idx = pd.date_range('2021-01-01', periods=48, freq='ME')
    trend_comp = np.linspace(100, 180, 48)
    seasonal_comp = 20 * np.sin(2*np.pi*np.arange(48)/12)
    noise = rng.normal(0, 5, 48)
    ts = pd.Series(trend_comp + seasonal_comp + noise, index=idx)

    result = seasonal_decompose(ts, model='additive', period=12)
    print('Decomposition components (first 6 months):')
    df = pd.DataFrame({
        'observed': result.observed,
        'trend':    result.trend,
        'seasonal': result.seasonal,
        'resid':    result.resid,
    }).dropna()
    print(df.head(6).round(2))
    print(f'\nTrend range: {result.trend.dropna().min():.1f} - {result.trend.dropna().max():.1f}')
    print(f'Seasonal amplitude: {result.seasonal.max():.1f}')
    print(f'Residual std: {result.resid.dropna().std():.2f}')
except ImportError:
    print('pip install statsmodels')

**Additive vs multiplicative decomposition**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    rng = np.random.default_rng(0)
    idx = pd.date_range('2020-01-01', periods=60, freq='ME')
    t = np.arange(60)
    trend = 100 + 2*t
    seasonal = 0.3 * np.sin(2*np.pi*t/12)  # proportional to trend

    additive_ts = pd.Series(trend + 20*np.sin(2*np.pi*t/12) + rng.normal(0,3,60), index=idx)
    multiplicative_ts = pd.Series(trend * (1 + seasonal) + rng.normal(0,3,60), index=idx)

    for name, ts, model in [('Additive TS', additive_ts, 'additive'),
                             ('Multiplicative TS', multiplicative_ts, 'multiplicative')]:
        dec = seasonal_decompose(ts, model=model, period=12)
        resid_std = dec.resid.dropna().std()
        print(f'{name} ({model}): residual std = {resid_std:.3f}')
        print(f'  Seasonal max = {dec.seasonal.max():.3f}')
except ImportError:
    print('pip install statsmodels')
    import numpy as np
    t = np.arange(60)
    ts = 100 + 2*t + 20*np.sin(2*np.pi*t/12) + np.random.normal(0,3,60)
    print('Manual trend (linear fit):')
    coeffs = np.polyfit(t, ts, 1)
    print(f'  slope={coeffs[0]:.2f}, intercept={coeffs[1]:.2f}')

**STL decomposition (robust to outliers)**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.seasonal import STL
    rng = np.random.default_rng(42)
    idx = pd.date_range('2020-01-01', periods=60, freq='ME')
    ts = pd.Series(
        100 + np.linspace(0, 40, 60) + 15*np.sin(2*np.pi*np.arange(60)/12) + rng.normal(0, 4, 60),
        index=idx
    )
    # Add outlier
    ts.iloc[24] = 250

    stl = STL(ts, period=12, robust=True)
    result = stl.fit()
    print('STL Decomposition (robust=True):')
    df = pd.DataFrame({'trend': result.trend, 'seasonal': result.seasonal, 'resid': result.resid})
    print(df.tail(6).round(2))
    print(f'\nMax residual (outlier detected at): {result.resid.abs().idxmax().date()}')
    print(f'Residual value: {result.resid.abs().max():.1f}')
except ImportError:
    print('pip install statsmodels')

**Extracting and using trend for forecasting**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(7)
idx = pd.date_range('2022-01-01', periods=36, freq='ME')
t = np.arange(36)
ts = pd.Series(100 + 3*t + 15*np.sin(2*np.pi*t/12) + rng.normal(0, 5, 36), index=idx)

# Fit linear trend
coeffs = np.polyfit(t, ts, 1)
trend  = np.polyval(coeffs, t)
detrended = ts.values - trend

# Seasonal component (average over each month)
seasonal = np.zeros(12)
for m in range(12):
    seasonal[m] = detrended[m::12].mean()

# Reconstruct and forecast next 6 months
t_fut = np.arange(36, 42)
trend_fut    = np.polyval(coeffs, t_fut)
seasonal_fut = seasonal[t_fut % 12]
forecast     = trend_fut + seasonal_fut

print(f'Trend: slope={coeffs[0]:.2f}/month, intercept={coeffs[1]:.2f}')
print('\nSeasonal pattern (monthly avg deviation):')
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for m, (name, val) in enumerate(zip(months, seasonal)):
    print(f'  {name}: {val:+.1f}')
print('\nForecast (next 6 months):')
fut_idx = pd.date_range('2025-01-01', periods=6, freq='ME')
for d, v in zip(fut_idx, forecast):
    print(f'  {d.strftime("%b %Y")}: {v:.1f}')

### Real-World Scenario

> An energy analyst decomposes hourly electricity consumption data to isolate the weekly seasonal pattern, long-term trend, and irregular spikes for demand forecasting.

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.seasonal import STL
    rng = np.random.default_rng(42)
    idx = pd.date_range('2024-01-01', periods=52*7, freq='D')
    t = np.arange(len(idx))
    trend_comp   = 500 + 0.3 * t
    weekly_seas  = 80 * np.sin(2*np.pi*t/7)
    annual_seas  = 120 * np.sin(2*np.pi*t/365)
    consumption  = pd.Series(trend_comp + weekly_seas + annual_seas + rng.normal(0, 15, len(t)), index=idx)

    stl = STL(consumption, period=7, robust=True)
    res = stl.fit()

    weekly_pattern = pd.Series(res.seasonal).groupby(pd.date_range('2024-01-01', periods=len(idx), freq='D').dayofweek).mean()
    days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    print('Weekly consumption pattern (avg deviation from trend):')
    for d, v in zip(days, weekly_pattern.reindex(range(7))):
        bar = '█' * int(abs(v) / 10)
        print(f'  {d}: {v:+.1f} kWh {bar}')
    print(f'\nOverall trend: +{(res.trend.iloc[-1] - res.trend.iloc[0]):.1f} kWh over the period')
    print(f'Residual std (noise): {res.resid.std():.1f} kWh')
except ImportError:
    print('pip install statsmodels')

### Practice: Decompose and Evaluate

Generate 3 years of monthly sales data with an upward trend and strong December seasonality (add 200 to December values). Use seasonal_decompose with period=12. Print the seasonal indices for each month. Which month has the highest seasonal component?

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2022-01-01', periods=36, freq='ME')
# TODO: create monthly sales with trend + December spike

# TODO: decompose

# TODO: print seasonal indices by month


## 6. 6. Stationarity & Differencing

Stationarity means constant mean, variance, and autocorrelation over time. Most statistical forecasting models require stationary input. Use the ADF test to check, and differencing to achieve stationarity.

**ADF test for stationarity**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.stattools import adfuller
    rng = np.random.default_rng(42)
    # Non-stationary: random walk
    rw = pd.Series(np.cumsum(rng.normal(0, 1, 100)))
    # Stationary: white noise
    wn = pd.Series(rng.normal(0, 1, 100))

    for name, ts in [('Random Walk', rw), ('White Noise', wn)]:
        result = adfuller(ts, autolag='AIC')
        print(f'{name}:')
        print(f'  ADF Statistic: {result[0]:.4f}')
        print(f'  p-value:       {result[1]:.4f}')
        print(f'  Critical 5%:   {result[4]["5%"]:.4f}')
        conclusion = 'STATIONARY' if result[1] < 0.05 else 'NON-STATIONARY'
        print(f'  Conclusion:    {conclusion}')
        print()
except ImportError:
    print('pip install statsmodels')
    import numpy as np
    rng = np.random.default_rng(42)
    ts = np.cumsum(rng.normal(0, 1, 100))
    # Simple manual check: variance of first vs second half
    print('Variance first half:', np.var(ts[:50]).round(2))
    print('Variance second half:', np.var(ts[50:]).round(2))

**First and second differencing**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.stattools import adfuller
    rng = np.random.default_rng(0)
    idx = pd.date_range('2020-01-01', periods=60, freq='ME')
    # Trend + noise (non-stationary)
    ts = pd.Series(100 + 2*np.arange(60) + rng.normal(0, 5, 60), index=idx)

    diff1 = ts.diff().dropna()
    diff2 = ts.diff().diff().dropna()

    for name, series in [('Original', ts), ('1st Diff', diff1), ('2nd Diff', diff2)]:
        adf = adfuller(series, autolag='AIC')
        stat = 'STATIONARY' if adf[1] < 0.05 else 'NON-STATIONARY'
        print(f'{name:10s}: ADF={adf[0]:7.3f}, p={adf[1]:.4f} → {stat}')

    print('\nDescriptive stats after 1st differencing:')
    print(diff1.describe().round(3))
except ImportError:
    print('pip install statsmodels')
    import numpy as np
    ts = np.cumsum(np.random.randn(60)) + np.arange(60)
    diff1 = np.diff(ts)
    print(f'Original mean: {ts.mean():.2f}, Diff mean: {diff1.mean():.2f}')
    print(f'Original std:  {ts.std():.2f}, Diff std:  {diff1.std():.2f}')

**Log transform for variance stabilization**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2020-01-01', periods=48, freq='ME')
# Growing variance (multiplicative process)
t = np.arange(48)
ts = pd.Series(100 * np.exp(0.05*t + rng.normal(0, 0.1, 48)), index=idx)

log_ts  = np.log(ts)
diff_ts = ts.diff().dropna()
log_diff = log_ts.diff().dropna()  # log returns

print('Original series stats:')
print(f'  Std first half: {ts[:24].std():.2f}')
print(f'  Std second half: {ts[24:].std():.2f}')
print('\nLog-transformed stats:')
print(f'  Std first half: {log_ts[:24].std():.4f}')
print(f'  Std second half: {log_ts[24:].std():.4f}')
print('\nLog-differenced (log returns) first 5:')
print(log_diff.head().round(4))
print(f'Annualized log return: {log_diff.mean() * 12:.3f}')

**Seasonal differencing**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.stattools import adfuller, kpss
    rng = np.random.default_rng(7)
    idx = pd.date_range('2020-01-01', periods=60, freq='ME')
    ts = pd.Series(
        100 + np.linspace(0, 20, 60) + 30*np.sin(2*np.pi*np.arange(60)/12) + rng.normal(0, 3, 60),
        index=idx
    )
    # Seasonal differencing (lag=12)
    seasonal_diff = ts.diff(12).dropna()
    # Then first difference to remove trend
    both_diff = seasonal_diff.diff().dropna()

    for name, s in [('Original', ts), ('Seasonal diff(12)', seasonal_diff), ('Both diffs', both_diff)]:
        adf_p = adfuller(s)[1]
        print(f'{name:20s}: ADF p={adf_p:.4f} → {"stationary" if adf_p < 0.05 else "non-stationary"}')
except ImportError:
    print('pip install statsmodels')
    import numpy as np
    ts = np.array([100 + 30*np.sin(2*np.pi*i/12) for i in range(60)])
    sdiff = ts[12:] - ts[:-12]
    print('Seasonal diff mean:', sdiff.mean().round(2))
    print('Seasonal diff std:', sdiff.std().round(2))

### Real-World Scenario

> A data scientist needs to prepare monthly website traffic data for ARIMA modeling — check stationarity, apply appropriate transformations, and verify the result.

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.stattools import adfuller
    rng = np.random.default_rng(42)
    idx = pd.date_range('2021-01-01', periods=48, freq='ME')
    traffic = pd.Series(
        5000 * np.exp(0.02 * np.arange(48)) +
        2000 * np.sin(2*np.pi*np.arange(48)/12) +
        rng.normal(0, 300, 48),
        index=idx, name='visitors'
    )

    def check_stationarity(series, name):
        adf = adfuller(series.dropna())
        stationary = adf[1] < 0.05
        print(f'{name}: p={adf[1]:.4f} | {"✓ stationary" if stationary else "✗ non-stationary"}')
        return stationary

    # Step 1: check original
    check_stationarity(traffic, 'Original')
    # Step 2: log transform
    log_traffic = np.log(traffic)
    check_stationarity(log_traffic, 'Log-transformed')
    # Step 3: seasonal diff
    log_sdiff = log_traffic.diff(12)
    check_stationarity(log_sdiff, 'Log + seasonal diff(12)')
    # Step 4: regular diff
    log_sdiff_diff = log_sdiff.diff()
    check_stationarity(log_sdiff_diff, 'Log + sdiff + diff')
    print('\nFinal series ready for ARIMA:')
    print(log_sdiff_diff.dropna().describe().round(4))
except ImportError:
    print('pip install statsmodels')

### Practice: Make It Stationary

Generate 60 months of quarterly sales data: base=500, trend=+5/month, seasonal amplitude=100, noise std=20. Apply the minimum number of transformations to make it stationary (ADF p < 0.05). Print the ADF result at each step.

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.stattools import adfuller
    rng = np.random.default_rng(0)
    idx = pd.date_range('2020-01-01', periods=60, freq='ME')
    # TODO: create series with trend + seasonality

    # TODO: apply transformations until stationary

    # TODO: print ADF result at each step
except ImportError:
    print('pip install statsmodels')

## 7. 7. ARIMA Modeling

ARIMA(p,d,q) combines AutoRegression (p lags), Integration (d differences for stationarity), and Moving Average (q lagged forecast errors). SARIMA extends this with seasonal components.

**Fitting ARIMA with statsmodels**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.arima.model import ARIMA
    rng = np.random.default_rng(42)
    # AR(1) process
    n = 100
    y = np.zeros(n)
    eps = rng.normal(0, 1, n)
    for t in range(1, n):
        y[t] = 0.7 * y[t-1] + eps[t]
    ts = pd.Series(y)

    model = ARIMA(ts, order=(1, 0, 0))
    result = model.fit()
    print(result.summary().tables[1])
    print(f'\nAIC: {result.aic:.2f}')
    print(f'BIC: {result.bic:.2f}')
    forecast = result.forecast(steps=5)
    print('\nForecast (next 5 steps):')
    print(forecast.round(3).tolist())
except ImportError:
    print('pip install statsmodels')

**Selecting ARIMA order with AIC**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.arima.model import ARIMA
    import warnings; warnings.filterwarnings('ignore')
    rng = np.random.default_rng(0)
    ts = pd.Series(np.cumsum(rng.normal(0, 1, 80)))  # random walk → d=1

    best_aic = float('inf')
    best_order = None
    results = []
    for p in range(3):
        for q in range(3):
            try:
                m = ARIMA(ts, order=(p, 1, q)).fit()
                results.append((p, 1, q, m.aic))
                if m.aic < best_aic:
                    best_aic, best_order = m.aic, (p, 1, q)
            except:
                pass
    results.sort(key=lambda x: x[3])
    print('Top 5 ARIMA orders by AIC:')
    for p, d, q, aic in results[:5]:
        mark = '<-- best' if (p,d,q) == best_order else ''
        print(f'  ARIMA({p},{d},{q}): AIC={aic:.2f} {mark}')
except ImportError:
    print('pip install statsmodels')

**ARIMA forecast with confidence intervals**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.arima.model import ARIMA
    import warnings; warnings.filterwarnings('ignore')
    rng = np.random.default_rng(42)
    idx = pd.date_range('2023-01-01', periods=60, freq='ME')
    ts = pd.Series(
        50 + np.linspace(0, 20, 60) + rng.normal(0, 4, 60),
        index=idx
    )
    train, test = ts[:48], ts[48:]
    model  = ARIMA(train, order=(1, 1, 1)).fit()
    fc     = model.get_forecast(steps=12)
    fc_df  = fc.summary_frame(alpha=0.05)

    print('Forecast vs Actual (last 6 months):')
    comparison = pd.DataFrame({
        'actual':   test.values,
        'forecast': fc_df['mean'].values,
        'lower_95': fc_df['mean_ci_lower'].values,
        'upper_95': fc_df['mean_ci_upper'].values,
    }, index=test.index)
    print(comparison.round(2))
    from sklearn.metrics import mean_absolute_error
    mae = mean_absolute_error(test, fc_df['mean'])
    print(f'\nMAE: {mae:.2f}')
except ImportError:
    print('pip install statsmodels scikit-learn')

**SARIMA for seasonal data**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    import warnings; warnings.filterwarnings('ignore')
    rng = np.random.default_rng(7)
    idx = pd.date_range('2020-01-01', periods=48, freq='ME')
    ts = pd.Series(
        100 + np.linspace(0, 20, 48) + 25*np.sin(2*np.pi*np.arange(48)/12) + rng.normal(0, 4, 48),
        index=idx
    )
    train = ts[:36]
    # SARIMA(1,1,1)(1,1,0,12)
    model  = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,0,12)).fit(disp=False)
    fc     = model.forecast(steps=12)
    actual = ts[36:]

    print('SARIMA Forecast vs Actual:')
    for d, f, a in zip(fc.index, fc.values, actual.values):
        err = f - a
        print(f'  {d.strftime("%b %Y")}: forecast={f:.1f}, actual={a:.1f}, error={err:+.1f}')
    rmse = np.sqrt(np.mean((fc.values - actual.values)**2))
    print(f'\nRMSE: {rmse:.2f}')
except ImportError:
    print('pip install statsmodels')

### Real-World Scenario

> A demand planner needs to forecast next 3 months of product demand using historical monthly data. They fit SARIMA, evaluate on a holdout set, and report RMSE and MAPE.

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    import warnings; warnings.filterwarnings('ignore')
    rng = np.random.default_rng(42)
    idx = pd.date_range('2021-01-01', periods=36, freq='ME')
    demand = pd.Series(
        500 + np.linspace(0, 100, 36) +
        150 * np.sin(2*np.pi*np.arange(36)/12) +
        rng.normal(0, 20, 36),
        index=idx
    ).clip(0)

    train, test = demand[:30], demand[30:]
    model  = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,0,12)).fit(disp=False)
    fc     = model.forecast(steps=6)

    mape = np.mean(np.abs((test.values - fc.values) / test.values)) * 100
    rmse = np.sqrt(np.mean((test.values - fc.values)**2))
    print('3-Month Demand Forecast:')
    for d, f, a in zip(fc.index, fc.values, test.values):
        print(f'  {d.strftime("%b %Y")}: forecast={f:.0f}, actual={a:.0f}')
    print(f'\nMAPE: {mape:.1f}%')
    print(f'RMSE: {rmse:.1f} units')
except ImportError:
    print('pip install statsmodels')

### Practice: Fit and Forecast

Generate 48 months of AR(2) data: y_t = 0.6*y_{t-1} - 0.2*y_{t-2} + noise. Use ARIMA(2,0,0) on the first 36 months. Forecast the last 12 months and compute MAE. Print forecast vs actual.

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.arima.model import ARIMA
    import warnings; warnings.filterwarnings('ignore')
    rng = np.random.default_rng(42)
    # TODO: generate AR(2) process

    # TODO: split train/test, fit ARIMA(2,0,0)

    # TODO: forecast and compute MAE
except ImportError:
    print('pip install statsmodels')

## 8. 8. Exponential Smoothing

Exponential smoothing methods weight recent observations more heavily. Simple ES handles level; Holt's adds trend; Holt-Winters adds seasonality. These are fast, interpretable, and competitive with ARIMA.

**Simple Exponential Smoothing**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.holtwinters import SimpleExpSmoothing
    rng = np.random.default_rng(42)
    ts = pd.Series(rng.normal(100, 10, 30))

    for alpha in [0.1, 0.3, 0.7]:
        model = SimpleExpSmoothing(ts, initialization_method='estimated').fit(smoothing_level=alpha, optimized=False)
        fc = model.forecast(3)
        print(f'alpha={alpha}: last fitted={model.fittedvalues.iloc[-1]:.2f}, forecast={fc.values.round(2).tolist()}')

    # Optimal alpha
    optimal = SimpleExpSmoothing(ts, initialization_method='estimated').fit(optimized=True)
    print(f'\nOptimal alpha: {optimal.params["smoothing_level"]:.4f}')
    print(f'Optimal forecast: {optimal.forecast(3).values.round(2).tolist()}')
except ImportError:
    print('pip install statsmodels')

**Holt's Linear Trend Method**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.holtwinters import Holt
    rng = np.random.default_rng(0)
    idx = pd.date_range('2024-01-01', periods=24, freq='ME')
    ts = pd.Series(50 + 2*np.arange(24) + rng.normal(0, 3, 24), index=idx)

    # Holt's with additive trend
    for trend in ['add', 'mul']:
        try:
            model = Holt(ts, exponential=(trend=='mul'), initialization_method='estimated').fit(optimized=True)
            fc    = model.forecast(6)
            print(f'Holt ({trend} trend):')
            print(f'  alpha={model.params["smoothing_level"]:.3f}, beta={model.params["smoothing_trend"]:.3f}')
            print(f'  Forecast: {fc.values.round(1).tolist()}')
        except:
            pass
    print('\nActual last 3:', ts.tail(3).values.round(1).tolist())
except ImportError:
    print('pip install statsmodels')

**Holt-Winters Triple Exponential Smoothing**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    import warnings; warnings.filterwarnings('ignore')
    rng = np.random.default_rng(42)
    idx = pd.date_range('2020-01-01', periods=48, freq='ME')
    ts = pd.Series(
        100 + np.linspace(0, 30, 48) + 25*np.sin(2*np.pi*np.arange(48)/12) + rng.normal(0, 4, 48),
        index=idx
    )
    train, test = ts[:36], ts[36:]

    model = ExponentialSmoothing(
        train, trend='add', seasonal='add', seasonal_periods=12,
        initialization_method='estimated'
    ).fit(optimized=True)

    fc = model.forecast(12)
    mape = np.mean(np.abs((test.values - fc.values) / test.values)) * 100
    rmse = np.sqrt(np.mean((test.values - fc.values)**2))
    print('Holt-Winters forecast:')
    print(f'  Alpha (level): {model.params["smoothing_level"]:.3f}')
    print(f'  Beta (trend):  {model.params["smoothing_trend"]:.3f}')
    print(f'  Gamma (season):{model.params["smoothing_seasonal"]:.3f}')
    print(f'\nMAPE: {mape:.2f}%')
    print(f'RMSE: {rmse:.2f}')
except ImportError:
    print('pip install statsmodels')

**Comparing ES methods on same data**

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
    import warnings; warnings.filterwarnings('ignore')
    rng = np.random.default_rng(7)
    idx = pd.date_range('2021-01-01', periods=36, freq='ME')
    ts = pd.Series(
        200 + np.linspace(0, 50, 36) + 30*np.sin(2*np.pi*np.arange(36)/12) + rng.normal(0, 8, 36),
        index=idx
    )
    train, test = ts[:30], ts[30:]
    n_fc = 6

    models = {
        'SES':   SimpleExpSmoothing(train, initialization_method='estimated').fit(optimized=True),
        'Holt':  Holt(train, initialization_method='estimated').fit(optimized=True),
        'HW':    ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12,
                                      initialization_method='estimated').fit(optimized=True),
    }
    print(f'{"Method":<8} {"RMSE":>8} {"MAPE%":>8}')
    print('-' * 28)
    for name, m in models.items():
        fc  = m.forecast(n_fc)
        rmse = np.sqrt(np.mean((test.values - fc.values)**2))
        mape = np.mean(np.abs((test.values - fc.values) / test.values)) * 100
        print(f'{name:<8} {rmse:>8.2f} {mape:>8.2f}')
except ImportError:
    print('pip install statsmodels')

### Real-World Scenario

> A retail company applies Holt-Winters to forecast daily store traffic for the next 30 days, accounting for weekly seasonality and a gradual upward trend.

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    import warnings; warnings.filterwarnings('ignore')
    rng = np.random.default_rng(42)
    idx = pd.date_range('2024-01-01', periods=90, freq='D')
    traffic = pd.Series(
        500 + np.linspace(0, 50, 90) +
        100 * np.sin(2*np.pi*np.arange(90)/7) +
        rng.normal(0, 20, 90),
        index=idx
    ).clip(0).round()

    train, test = traffic[:75], traffic[75:]
    model = ExponentialSmoothing(
        train, trend='add', seasonal='add', seasonal_periods=7,
        initialization_method='estimated'
    ).fit(optimized=True)

    fc = model.forecast(15)
    mae  = np.mean(np.abs(test.values - fc.values))
    mape = np.mean(np.abs((test.values - fc.values) / test.values)) * 100

    print('Holt-Winters Daily Traffic Forecast:')
    print(f'  Params: alpha={model.params["smoothing_level"]:.3f}, '
          f'beta={model.params["smoothing_trend"]:.3f}, '
          f'gamma={model.params["smoothing_seasonal"]:.3f}')
    print(f'\nForecast (next 15 days):')
    days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    for d, f, a in zip(fc.index, fc.values, test.values):
        print(f'  {d.strftime("%a %b %d")}: forecast={f:.0f}, actual={a:.0f}')
    print(f'\nMAE: {mae:.1f} | MAPE: {mape:.1f}%')
except ImportError:
    print('pip install statsmodels')

### Practice: Holt-Winters Tuning

Generate 3 years of monthly retail sales with trend and seasonality. Compare Holt-Winters additive vs multiplicative seasonal on a 6-month holdout. Print RMSE for both. Which model wins?

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    import warnings; warnings.filterwarnings('ignore')
    rng = np.random.default_rng(42)
    idx = pd.date_range('2021-01-01', periods=36, freq='ME')
    # TODO: create monthly sales with additive or multiplicative seasonality

    # TODO: split and fit both models

    # TODO: compare RMSE
except ImportError:
    print('pip install statsmodels')

## 9. 9. Feature Engineering for Time Series

Transforming raw time series into supervised learning features enables ML models (XGBoost, LightGBM) to forecast. Key features: lag values, rolling stats, calendar attributes, and cyclical encoding.

**Lag features**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', periods=30, freq='D')
ts = pd.Series(rng.normal(100, 10, 30), index=idx, name='demand')
df = ts.to_frame()

# Create lag features
for lag in [1, 2, 3, 7, 14]:
    df[f'lag_{lag}'] = df['demand'].shift(lag)

df.dropna(inplace=True)
print(f'Feature matrix shape: {df.shape}')
print(df.head(5).round(2))

# Correlation with target
corr = df.corr()['demand'].drop('demand')
print('\nLag correlations with demand:')
print(corr.round(3))

**Rolling statistical features**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(0)
idx = pd.date_range('2024-01-01', periods=60, freq='D')
ts = pd.Series(rng.normal(500, 80, 60) + np.linspace(0, 50, 60), index=idx, name='sales')
df = ts.to_frame()

# Rolling features
for w in [7, 14]:
    roll = ts.rolling(w)
    df[f'roll_{w}_mean'] = roll.mean()
    df[f'roll_{w}_std']  = roll.std()
    df[f'roll_{w}_min']  = roll.min()
    df[f'roll_{w}_max']  = roll.max()

# Expanding mean
df['expanding_mean'] = ts.expanding().mean()

df.dropna(inplace=True)
print(f'Feature shape: {df.shape}')
print(df.tail(5).round(1))

**Calendar and cyclical features**

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(7)
idx = pd.date_range('2024-01-01', periods=365, freq='D')
df = pd.DataFrame({'sales': rng.normal(100, 15, 365)}, index=idx)

# Calendar features
df['day_of_week']  = df.index.dayofweek
df['day_of_month'] = df.index.day
df['month']        = df.index.month
df['quarter']      = df.index.quarter
df['is_weekend']   = (df.index.dayofweek >= 5).astype(int)
df['is_month_end'] = df.index.is_month_end.astype(int)

# Cyclical encoding (preserves periodicity)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 12)
df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 12)

print('Feature matrix (first 5 rows):')
print(df.head(5).round(3))
print(f'\nTotal features: {df.shape[1]-1}')
print('Weekend mean:', df[df['is_weekend']==1]['sales'].mean().round(2))
print('Weekday mean:', df[df['is_weekend']==0]['sales'].mean().round(2))

**ML forecasting pipeline with lag features**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

rng = np.random.default_rng(42)
idx = pd.date_range('2023-01-01', periods=200, freq='D')
ts = pd.Series(
    100 + np.linspace(0, 30, 200) + 20*np.sin(2*np.pi*np.arange(200)/7) + rng.normal(0, 5, 200),
    index=idx, name='demand'
)

df = ts.to_frame()
# Lag + rolling features
for lag in [1, 2, 3, 7]: df[f'lag_{lag}'] = ts.shift(lag)
df['roll7_mean'] = ts.rolling(7).mean()
df['roll7_std']  = ts.rolling(7).std()
df['dow'] = ts.index.dayofweek
df['dow_sin'] = np.sin(2*np.pi*df['dow']/7)
df['dow_cos'] = np.cos(2*np.pi*df['dow']/7)
df.dropna(inplace=True)

X = df.drop(columns='demand')
y = df['demand']
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

model = GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

mae  = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f'GBM Forecast: MAE={mae:.2f}, RMSE={rmse:.2f}')
print('\nTop feature importances:')
fi = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
for feat, imp in fi.head(5).items():
    print(f'  {feat:15s}: {imp:.4f}')

### Real-World Scenario

> A logistics company builds an ML forecasting model for package delivery volumes using lag features, rolling statistics, and calendar effects as inputs to a gradient boosting model.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(42)
idx = pd.date_range('2023-01-01', periods=365, freq='D')
volume = pd.Series(
    3000 + np.linspace(0, 500, 365) +
    800 * np.sin(2*np.pi*np.arange(365)/7) +
    500 * (idx.month == 12).astype(float) +  # December peak
    rng.normal(0, 100, 365),
    index=idx
).clip(0)

df = volume.to_frame(name='volume')
for lag in [1, 2, 3, 7, 14]: df[f'lag_{lag}'] = volume.shift(lag)
for w in [7, 14]: df[f'roll_{w}_mean'] = volume.rolling(w).mean()
df['dow']       = volume.index.dayofweek
df['month']     = volume.index.month
df['is_weekend']= (df['dow'] >= 5).astype(int)
df['dow_sin']   = np.sin(2*np.pi*df['dow']/7)
df['dow_cos']   = np.cos(2*np.pi*df['dow']/7)
df.dropna(inplace=True)

X, y = df.drop('volume', axis=1), df['volume']
split = 300
model = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
model.fit(X.iloc[:split], y.iloc[:split])
preds = model.predict(X.iloc[split:])
mae = mean_absolute_error(y.iloc[split:], preds)
print(f'MAE: {mae:.0f} packages/day')
fi = pd.Series(model.feature_importances_, index=X.columns).nlargest(5)
print('\nTop features:')
for f, v in fi.items():
    print(f'  {f}: {v:.4f}')

### Practice: Build a Feature Matrix

Use the hourly energy dataset (simulate: 7 days × 24 hours = 168 hourly readings with daily cycle). Create: lag_1, lag_24 (same hour yesterday), roll_24_mean, hour_of_day, is_weekday. Train a LinearRegression model and print RMSE.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', periods=168, freq='h')
# TODO: create hourly energy with daily cycle

# TODO: build feature matrix

# TODO: train LinearRegression and print RMSE


## 10. 10. Forecasting Evaluation

Proper evaluation of time series forecasts requires time-ordered splits (no leakage), multiple metrics (MAE, RMSE, MAPE), and ideally walk-forward validation to simulate real deployment.

**MAE, RMSE, MAPE, and SMAPE**

In [ ]:
import numpy as np

actual   = np.array([100, 120, 130, 115, 140, 125, 160, 155, 170, 180])
forecast = np.array([105, 115, 135, 110, 145, 130, 150, 160, 165, 185])

mae   = np.mean(np.abs(actual - forecast))
mse   = np.mean((actual - forecast)**2)
rmse  = np.sqrt(mse)
mape  = np.mean(np.abs((actual - forecast) / actual)) * 100
smape = np.mean(2 * np.abs(actual - forecast) / (np.abs(actual) + np.abs(forecast))) * 100
# R-squared
ss_res = np.sum((actual - forecast)**2)
ss_tot = np.sum((actual - actual.mean())**2)
r2 = 1 - ss_res / ss_tot

print(f'MAE:   {mae:.2f}')
print(f'RMSE:  {rmse:.2f}')
print(f'MAPE:  {mape:.2f}%')
print(f'sMAPE: {smape:.2f}%')
print(f'R²:    {r2:.4f}')
print(f'\nBias (mean error): {(forecast - actual).mean():.2f}')

**Train/test split for time series**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge

rng = np.random.default_rng(42)
idx = pd.date_range('2023-01-01', periods=200, freq='D')
ts  = pd.Series(100 + np.linspace(0,30,200) + rng.normal(0, 8, 200), index=idx)

df = ts.to_frame(name='y')
for lag in [1, 7, 14]: df[f'lag_{lag}'] = ts.shift(lag)
df.dropna(inplace=True)

X = df.drop('y', axis=1).values
y = df['y'].values

# Time-ordered split (NEVER random split for time series!)
n = len(X)
split = int(n * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = Ridge().fit(X_train, y_train)
preds = model.predict(X_test)
mae   = np.mean(np.abs(y_test - preds))
rmse  = np.sqrt(np.mean((y_test - preds)**2))
print(f'Train size: {split}, Test size: {n - split}')
print(f'MAE:  {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print('\nNEVER use random train_test_split for time series!')
print('This leaks future information into training data.')

**Walk-forward (expanding window) validation**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge

rng = np.random.default_rng(42)
n   = 120
ts  = pd.Series(100 + np.linspace(0, 20, n) + rng.normal(0, 6, n))

df = ts.to_frame(name='y')
for lag in [1, 2, 3, 7]: df[f'lag_{lag}'] = ts.shift(lag)
df.dropna(inplace=True)

X = df.drop('y', axis=1).values
y = df['y'].values

# Walk-forward validation
min_train = 30
step = 5
errors = []
for test_start in range(min_train, len(X) - step + 1, step):
    X_train, y_train = X[:test_start], y[:test_start]
    X_test,  y_test  = X[test_start:test_start+step], y[test_start:test_start+step]
    model = Ridge().fit(X_train, y_train)
    preds = model.predict(X_test)
    errors.extend(np.abs(y_test - preds).tolist())

print(f'Walk-forward CV ({len(errors)} predictions):')
print(f'  MAE:  {np.mean(errors):.2f}')
print(f'  RMSE: {np.sqrt(np.mean(np.array(errors)**2)):.2f}')
print(f'  Std:  {np.std(errors):.2f}')

**TimeSeriesSplit cross-validation with sklearn**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor

rng = np.random.default_rng(42)
n   = 200
ts  = pd.Series(100 + np.linspace(0,40,n) + rng.normal(0,8,n))

df = ts.to_frame(name='y')
for lag in [1,2,3,7]: df[f'lag_{lag}'] = ts.shift(lag)
df['roll7'] = ts.rolling(7).mean()
df.dropna(inplace=True)

X = df.drop('y', axis=1)
y = df['y']

tscv = TimeSeriesSplit(n_splits=5, gap=0)
model = GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=42)
scores = cross_val_score(model, X, y, cv=tscv, scoring='neg_mean_absolute_error')
maes = -scores

print('TimeSeriesSplit CV results (5 folds):')
for i, (mae, (tr, te)) in enumerate(zip(maes, tscv.split(X)), 1):
    print(f'  Fold {i}: train={len(tr)}, test={len(te)}, MAE={mae:.2f}')
print(f'\nMean MAE: {maes.mean():.2f} ± {maes.std():.2f}')

### Real-World Scenario

> A data science team evaluates three forecasting models (ARIMA, Holt-Winters, GBM) on 1 year of daily data using walk-forward validation to select the production model.

In [ ]:
import pandas as pd
import numpy as np

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    from statsmodels.tsa.arima.model import ARIMA
    from sklearn.ensemble import GradientBoostingRegressor
    import warnings; warnings.filterwarnings('ignore')

    rng = np.random.default_rng(42)
    n   = 200
    idx = pd.date_range('2024-01-01', periods=n, freq='D')
    ts  = pd.Series(
        300 + np.linspace(0,50,n) + 60*np.sin(2*np.pi*np.arange(n)/7) + rng.normal(0,15,n),
        index=idx
    )

    train, test = ts[:160], ts[160:]
    results = {}

    # Holt-Winters
    hw = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=7, initialization_method='estimated').fit(optimized=True)
    results['Holt-Winters'] = hw.forecast(40)

    # GBM with lag features
    df = ts.to_frame(name='y')
    for lag in [1,2,7]: df[f'lag_{lag}'] = ts.shift(lag)
    df.dropna(inplace=True)
    X, y = df.drop('y',axis=1), df['y']
    split = len(train) - 7  # adjust for lag
    gbm = GradientBoostingRegressor(n_estimators=100, random_state=42)
    gbm.fit(X.iloc[:split], y.iloc[:split])
    results['GBM'] = pd.Series(gbm.predict(X.iloc[split:split+40]), index=test.index[:40])

    actual = test.values[:40]
    print('Model Comparison (40-day holdout):')
    print(f'{"Model":<15} {"MAE":>7} {"RMSE":>7} {"MAPE%":>7}')
    print('-' * 38)
    for name, fc in results.items():
        fc_v = fc.values[:40]
        mae  = np.mean(np.abs(actual - fc_v))
        rmse = np.sqrt(np.mean((actual - fc_v)**2))
        mape = np.mean(np.abs((actual - fc_v)/actual))*100
        print(f'{name:<15} {mae:>7.1f} {rmse:>7.1f} {mape:>7.1f}')
except ImportError:
    print('pip install statsmodels scikit-learn')

### Practice: Cross-Validate a Forecast

Generate 180 days of daily demand data with weekly seasonality. Use TimeSeriesSplit with 5 folds on lag features. Test Ridge, GradientBoosting, and a naive (last value) baseline. Print mean MAE per model.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor

rng = np.random.default_rng(42)
idx = pd.date_range('2024-01-01', periods=180, freq='D')
# TODO: create demand series

# TODO: create feature matrix

# TODO: cross-validate all three models


## 11. 11. Prophet Forecasting

Use Facebook Prophet for additive time series forecasting with automatic trend, seasonality, and holiday effects. Handle changepoints and produce uncertainty intervals.

**Basic Prophet fit and forecast**

In [ ]:
import pandas as pd
import numpy as np

# Simulate daily data with trend + weekly seasonality (Prophet-ready format)
np.random.seed(42)
dates = pd.date_range('2022-01-01', periods=365, freq='D')
trend = np.linspace(100, 200, 365)
weekly = 10 * np.sin(2 * np.pi * np.arange(365) / 7)
noise = np.random.normal(0, 5, 365)
y = trend + weekly + noise

df = pd.DataFrame({'ds': dates, 'y': y})
print('Prophet input format:')
print(df.head())
print(f'\nShape: {df.shape}')
print(f'Date range: {df.ds.min().date()} to {df.ds.max().date()}')
print(f'y stats: mean={df.y.mean():.1f}, std={df.y.std():.1f}')

# Without Prophet installed, demonstrate the workflow:
print('\nProphet workflow:')
print('  from prophet import Prophet')
print('  m = Prophet(yearly_seasonality=False, weekly_seasonality=True)')
print('  m.fit(df)')
print('  future = m.make_future_dataframe(periods=30)')
print('  forecast = m.predict(future)')
print('  m.plot(forecast)')

**Manual additive decomposition as Prophet substitute**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)
dates = pd.date_range('2022-01-01', periods=365, freq='D')
trend = np.linspace(100, 200, 365)
weekly = 8 * np.sin(2 * np.pi * np.arange(365) / 7)
noise = np.random.normal(0, 5, 365)
y = trend + weekly + noise

df = pd.DataFrame({'ds': dates, 'y': y})
df['dow'] = df['ds'].dt.dayofweek

# Fit trend with linear regression
from sklearn.linear_model import LinearRegression
t = np.arange(len(df)).reshape(-1, 1)
lr = LinearRegression().fit(t, df['y'])
df['trend_fit'] = lr.predict(t)

# Fit weekly seasonality as day-of-week means on residuals
df['resid'] = df['y'] - df['trend_fit']
dow_effect = df.groupby('dow')['resid'].mean()
df['seasonal'] = df['dow'].map(dow_effect)
df['yhat'] = df['trend_fit'] + df['seasonal']

mse = ((df['y'] - df['yhat'])**2).mean()**0.5
print(f'RMSE (train): {mse:.2f}')
fig, axes = plt.subplots(3, 1, figsize=(10, 8))
axes[0].plot(df['ds'], df['y'], alpha=0.5, label='actual'); axes[0].plot(df['ds'], df['yhat'], label='fit'); axes[0].legend(); axes[0].set_title('Additive Model Fit')
axes[1].plot(df['ds'], df['trend_fit']); axes[1].set_title('Trend Component')
axes[2].bar(range(7), dow_effect.values); axes[2].set_xticks(range(7)); axes[2].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun']); axes[2].set_title('Weekly Seasonality')
plt.tight_layout(); plt.savefig('prophet_decomp.png', dpi=80); plt.close(); print('Saved prophet_decomp.png')

**Changepoint detection with PELT algorithm**

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Simulate series with structural breaks
np.random.seed(42)
n = 300
t = np.arange(n)
# Three regimes: flat, upward, downward
y = np.concatenate([
    np.random.normal(10, 2, 100),
    np.random.normal(25, 2, 100),
    np.random.normal(15, 2, 100)
])

# Simple CUSUM-based changepoint detection
def cusum_changepoints(series, threshold=10):
    mu = series.mean()
    cusum_pos = np.zeros(len(series))
    cusum_neg = np.zeros(len(series))
    changepoints = []
    for i in range(1, len(series)):
        cusum_pos[i] = max(0, cusum_pos[i-1] + series[i] - mu - 0.5)
        cusum_neg[i] = max(0, cusum_neg[i-1] - series[i] + mu - 0.5)
        if cusum_pos[i] > threshold or cusum_neg[i] > threshold:
            changepoints.append(i)
            cusum_pos[i] = cusum_neg[i] = 0
    return changepoints

cps = cusum_changepoints(y, threshold=15)
print(f'Detected changepoints at indices: {cps}')
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, y, alpha=0.7)
for cp in cps:
    ax.axvline(cp, color='red', linestyle='--', linewidth=2, label=f'CP at {cp}' if cp == cps[0] else '')
ax.set_title('CUSUM Changepoint Detection'); ax.legend()
plt.tight_layout(); plt.savefig('changepoints.png', dpi=80); plt.close(); print('Saved changepoints.png')

**Cross-validation for forecasting (walk-forward)**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge

np.random.seed(42)
dates = pd.date_range('2022-01-01', periods=200, freq='D')
trend = np.linspace(0, 50, 200)
weekly = 5 * np.sin(2 * np.pi * np.arange(200) / 7)
noise = np.random.normal(0, 3, 200)
y = trend + weekly + noise

df = pd.DataFrame({'ds': dates, 'y': y})
df['t'] = np.arange(len(df))
df['dow'] = df['ds'].dt.dayofweek
# Encode day-of-week as dummies
for d in range(7):
    df[f'dow_{d}'] = (df['dow'] == d).astype(int)

feat_cols = ['t'] + [f'dow_{d}' for d in range(7)]
X = df[feat_cols].values

# Walk-forward validation: train on first 60%, then expand
initial = 120; horizon = 7; errors = []
for start in range(initial, len(df) - horizon, horizon):
    X_tr, y_tr = X[:start], y[:start]
    X_te, y_te = X[start:start+horizon], y[start:start+horizon]
    m = Ridge().fit(X_tr, y_tr)
    preds = m.predict(X_te)
    errors.append(np.sqrt(((y_te - preds)**2).mean()))

print(f'Walk-forward validation ({len(errors)} folds):')
print(f'  Mean RMSE: {np.mean(errors):.2f}')
print(f'  Std  RMSE: {np.std(errors):.2f}')
print(f'  Min  RMSE: {min(errors):.2f}, Max: {max(errors):.2f}')

### Real-World Scenario

> Forecast retail store daily sales for next 30 days using historical 2 years of data. The data has weekly patterns (lower on Mondays, peak on weekends), a monthly pay-cycle boost, and a gradual upward trend. Produce point forecasts and 80% prediction intervals.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

np.random.seed(0)
dates = pd.date_range('2022-01-01', periods=730, freq='D')
trend = np.linspace(500, 700, 730)
weekly = 40 * np.sin(2 * np.pi * np.arange(730) / 7 - np.pi)
monthly = 30 * np.sin(2 * np.pi * np.arange(730) / 30)
noise = np.random.normal(0, 15, 730)
y = trend + weekly + monthly + noise

df = pd.DataFrame({'ds': dates, 'y': y})
df['t'] = np.arange(len(df))
df['dow'] = df['ds'].dt.dayofweek
df['dom'] = df['ds'].dt.day
for d in range(7): df[f'dow_{d}'] = (df['dow'] == d).astype(int)

feat_cols = ['t'] + [f'dow_{d}' for d in range(7)] + ['dom']
X = df[feat_cols].values
train = df[df['ds'] < '2023-07-01']; test = df[df['ds'] >= '2023-07-01']
idx_tr = df['ds'] < '2023-07-01'

sc = StandardScaler().fit(X[idx_tr])
m = Ridge(alpha=1.0).fit(sc.transform(X[idx_tr]), y[idx_tr])

resid = y[idx_tr] - m.predict(sc.transform(X[idx_tr]))
pred = m.predict(sc.transform(X[~idx_tr]))
print(f'Forecast RMSE: {np.sqrt(((y[~idx_tr]-pred)**2).mean()):.2f}')
print(f'80% PI: ±{1.28*resid.std():.2f}')

### Practice: Forecast Weekly Revenue with Confidence Intervals

Given 2 years of weekly revenue data with trend and seasonality, build a feature-based Ridge model (lag features + time index + month dummies). Perform walk-forward validation with 4-week horizons. Report average RMSE and generate 80% prediction intervals using residual std.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge

np.random.seed(7)
weeks = pd.date_range('2022-01-03', periods=104, freq='W')
trend = np.linspace(10000, 15000, 104)
seasonality = 1500 * np.sin(2 * np.pi * np.arange(104) / 52)
noise = np.random.normal(0, 300, 104)
y = trend + seasonality + noise
df = pd.DataFrame({'ds': weeks, 'revenue': y})

# TODO: Create features (t index, month dummies, lag-1)
# TODO: Walk-forward cross-validation with 4-week horizon
# TODO: Report average RMSE and 80% PI width


## 12. 12. Anomaly Detection in Time Series

Detect outliers and anomalies in sequential data using statistical thresholds, isolation forests, and autoencoder-style reconstruction errors.

**Z-score and IQR rolling anomaly detection**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=100, freq='D')
y = np.sin(np.arange(100) * 0.3) * 10 + np.random.normal(0, 1, 100) + 50
# Inject anomalies
y[[20, 45, 70, 85]] += [25, -20, 30, -15]

df = pd.DataFrame({'ds': dates, 'y': y})

# Rolling z-score
win = 14
df['roll_mean'] = df['y'].rolling(win, min_periods=1).mean()
df['roll_std']  = df['y'].rolling(win, min_periods=1).std().fillna(1)
df['zscore']    = (df['y'] - df['roll_mean']) / df['roll_std']
df['anomaly_z'] = df['zscore'].abs() > 2.5

print(f'Z-score anomalies: {df["anomaly_z"].sum()} detected')
print(df[df['anomaly_z']][['ds','y','zscore']].to_string())
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df['ds'], df['y'], label='series')
ax.scatter(df[df['anomaly_z']]['ds'], df[df['anomaly_z']]['y'],
           color='red', s=80, zorder=5, label='anomaly')
ax.set_title('Rolling Z-score Anomaly Detection'); ax.legend()
plt.tight_layout(); plt.savefig('anomaly_zscore.png', dpi=80); plt.close(); print('Saved anomaly_zscore.png')

**Isolation Forest for multivariate anomalies**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)
n = 200
dates = pd.date_range('2024-01-01', periods=n, freq='H')
cpu    = np.random.normal(40, 10, n)
memory = cpu * 1.5 + np.random.normal(0, 5, n)
# Inject anomalies
cpu[[50, 100, 150]] = [95, 5, 98]
memory[[50, 100, 150]] = [30, 120, 25]

df = pd.DataFrame({'ds': dates, 'cpu': cpu, 'memory': memory})
X = df[['cpu', 'memory']].values

iso = IsolationForest(contamination=0.05, random_state=42)
df['anomaly'] = iso.fit_predict(X) == -1
df['score']   = iso.score_samples(X)

print(f'Anomalies detected: {df["anomaly"].sum()}')
print(df[df['anomaly']][['ds','cpu','memory','score']].to_string())
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
for i, col in enumerate(['cpu', 'memory']):
    axes[i].plot(df['ds'], df[col], alpha=0.7)
    axes[i].scatter(df[df['anomaly']]['ds'], df[df['anomaly']][col],
                    color='red', s=60, zorder=5)
    axes[i].set_ylabel(col)
plt.suptitle('Isolation Forest Anomaly Detection'); plt.tight_layout()
plt.savefig('anomaly_iforest.png', dpi=80); plt.close(); print('Saved anomaly_iforest.png')

**LSTM autoencoder reconstruction error**

In [ ]:
import numpy as np

np.random.seed(42)
# Simulate normal signal
t = np.linspace(0, 20*np.pi, 500)
normal = np.sin(t) + 0.1 * np.random.randn(500)
# Inject anomaly window
anom = normal.copy()
anom[200:220] += 5  # spike

# Simple threshold-based reconstruction error (PCA substitute)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Create sliding windows
def make_windows(x, w=20):
    return np.array([x[i:i+w] for i in range(len(x)-w)])

X = make_windows(normal, 20)
X_anom = make_windows(anom, 20)

sc = StandardScaler().fit(X)
X_sc = sc.transform(X)
X_anom_sc = sc.transform(X_anom)

pca = PCA(n_components=5).fit(X_sc)
rec = pca.inverse_transform(pca.transform(X_anom_sc))
errors = ((X_anom_sc - rec)**2).mean(axis=1)

threshold = np.percentile(errors, 95)
print(f'Reconstruction error threshold (95th pct): {threshold:.4f}')
print(f'Anomaly windows detected: {(errors > threshold).sum()}')
print(f'Max error at index: {errors.argmax()} (injected at 200:220)')

**STL-based anomaly detection**

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import STL
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')
trend = np.linspace(0, 30, 365)
seasonal = 10 * np.sin(2 * np.pi * np.arange(365) / 7)
noise = np.random.normal(0, 2, 365)
y = trend + seasonal + noise

# Inject point anomalies
injected = [80, 160, 250, 320]
y_anom = y.copy()
for idx in injected:
    y_anom[idx] += np.random.choice([-25, 25])

series = pd.Series(y_anom, index=dates)
stl = STL(series, period=7, robust=True)
result = stl.fit()
residual = result.resid

thresh = 3 * residual.std()
anomalies = series[residual.abs() > thresh]
print(f'STL anomalies detected: {len(anomalies)}')
print(f'Injected at: {injected}')
print(f'Detected at: {list(anomalies.index.dayofyear)}')
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(series.index, series, alpha=0.6); axes[0].scatter(anomalies.index, anomalies, color='red', s=60, zorder=5, label='anomaly'); axes[0].legend(); axes[0].set_title('STL Anomaly Detection')
axes[1].plot(residual.index, residual); axes[1].axhline(thresh, color='r', linestyle='--'); axes[1].axhline(-thresh, color='r', linestyle='--'); axes[1].set_title('STL Residual')
plt.tight_layout(); plt.savefig('stl_anomaly.png', dpi=80); plt.close(); print('Saved stl_anomaly.png')

### Real-World Scenario

> Monitor server CPU usage (sampled every 5 minutes) to automatically alert when anomalies occur. The signal has daily patterns (low at night, high during business hours). You must detect both sudden spikes and sustained elevated periods while minimizing false alarms.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

np.random.seed(1)
# 2 weeks of 5-min samples = 4032 points
t = np.arange(4032)
daily = 20 * np.sin(2*np.pi*t / 288 - np.pi/2) + 50  # 288 samples/day
noise = np.random.normal(0, 3, 4032)
cpu = np.clip(daily + noise, 5, 100)
# Inject anomalies
cpu[500:510]  += 40  # spike
cpu[1200:1230] += 30  # sustained high

times = pd.date_range('2024-01-01', periods=4032, freq='5min')
df = pd.DataFrame({'time': times, 'cpu': cpu})
df['hour'] = df['time'].dt.hour
df['roll_mean'] = df['cpu'].rolling(12).mean().fillna(method='bfill')
df['roll_std']  = df['cpu'].rolling(12).std().fillna(1)

# Isolation Forest on features
X = df[['cpu', 'hour', 'roll_mean', 'roll_std']].values
iso = IsolationForest(contamination=0.01, random_state=42)
df['anomaly'] = iso.fit_predict(X) == -1
print(f'Anomalies detected: {df["anomaly"].sum()} of {len(df)} points ({df["anomaly"].mean():.1%})')
print(df[df['anomaly']].groupby(df[df['anomaly']]['time'].dt.date)['anomaly'].count())

### Practice: Detect Order Volume Anomalies

Given hourly e-commerce order counts with weekly seasonality, detect anomalies using rolling z-score (window=24h, threshold=3). Also apply Isolation Forest on (count, hour, day_of_week) features. Compare which method catches more of the injected spikes (3 injected anomalies) with fewer false positives.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

np.random.seed(99)
hours = pd.date_range('2024-01-01', periods=336, freq='H')  # 2 weeks
base = 100 + 50 * np.sin(2*np.pi*np.arange(336)/24 - np.pi)
noise = np.random.normal(0, 8, 336)
orders = np.clip(base + noise, 0, None)
orders[[100, 200, 280]] += [150, -80, 200]  # inject anomalies
df = pd.DataFrame({'time': hours, 'orders': orders})

# TODO: Rolling z-score anomaly detection (window=24, thresh=3)
# TODO: Isolation Forest on (orders, hour, dayofweek)
# TODO: Compare detected anomaly counts and false positive rates


## 13. 13. Neural Forecasting Fundamentals

Apply LSTM, Temporal Convolutional Networks (TCN), and N-BEATS-style architectures using PyTorch or Keras for sequence forecasting tasks.

**Univariate LSTM forecasting (NumPy / PyTorch-free)**

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Demonstrate the lag-feature approach that underlies RNN forecasting
np.random.seed(42)
n = 300
t = np.arange(n)
y = np.sin(0.2*t) * 10 + 0.1*t + np.random.normal(0, 1, n)

def make_supervised(series, lags=10):
    X, Y = [], []
    for i in range(lags, len(series)):
        X.append(series[i-lags:i])
        Y.append(series[i])
    return np.array(X), np.array(Y)

X, Y = make_supervised(y, lags=20)
split = int(0.8 * len(X))
X_tr, X_te = X[:split], X[split:]
Y_tr, Y_te = Y[:split], Y[split:]

sc_x = StandardScaler().fit(X_tr)
sc_y = StandardScaler().fit(Y_tr.reshape(-1,1))

model = Ridge(alpha=0.1)
model.fit(sc_x.transform(X_tr), sc_y.transform(Y_tr.reshape(-1,1)).ravel())

pred_sc = model.predict(sc_x.transform(X_te))
pred = sc_y.inverse_transform(pred_sc.reshape(-1,1)).ravel()

rmse = np.sqrt(((Y_te - pred)**2).mean())
print(f'Lag-20 linear model RMSE: {rmse:.3f}')
print(f'Baseline (mean) RMSE:     {np.sqrt(((Y_te - Y_tr.mean())**2).mean()):.3f}')
print('This is the feature-engineering equivalent of an LSTM cell state.')

**Sequence-to-sequence multi-step forecasting**

In [ ]:
import numpy as np
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import GradientBoostingRegressor

np.random.seed(42)
n = 400
t = np.arange(n)
y = 20 * np.sin(0.1*t) + 0.05*t + np.random.normal(0, 2, n)

def make_seq2seq(series, lookback=30, horizon=7):
    X, Y = [], []
    for i in range(lookback, len(series) - horizon + 1):
        X.append(series[i-lookback:i])
        Y.append(series[i:i+horizon])
    return np.array(X), np.array(Y)

X, Y = make_seq2seq(y, lookback=30, horizon=7)
split = int(0.8 * len(X))
X_tr, X_te = X[:split], X[split:]
Y_tr, Y_te = Y[:split], Y[split:]

# Multi-output regression (equivalent to decoder in seq2seq)
model = MultiOutputRegressor(GradientBoostingRegressor(n_estimators=50, random_state=0))
model.fit(X_tr, Y_tr)
pred = model.predict(X_te)

rmse_by_step = np.sqrt(((Y_te - pred)**2).mean(axis=0))
print('RMSE by forecast horizon step:')
for step, r in enumerate(rmse_by_step, 1):
    print(f'  Step {step}: {r:.3f}')
print(f'Overall RMSE: {rmse_by_step.mean():.3f}')

**Attention-weighted feature importance for time series**

In [ ]:
import numpy as np

np.random.seed(42)
n = 500
t = np.arange(n)
y = 15*np.sin(0.15*t) + 5*np.sin(0.05*t) + np.random.normal(0, 1, n)

def make_windows(series, w=20):
    X = np.array([series[i-w:i] for i in range(w, len(series))])
    Y = series[w:]
    return X, Y

X, Y = make_windows(y, 20)

# Simulate attention: weight lags by their correlation with target
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

sc_x = StandardScaler().fit(X)
sc_y = StandardScaler().fit(Y.reshape(-1,1))
X_sc = sc_x.transform(X)
Y_sc = sc_y.transform(Y.reshape(-1,1)).ravel()

# Attention weights = absolute Ridge coefficients (softmax)
model = Ridge(alpha=0.01).fit(X_sc, Y_sc)
raw_weights = np.abs(model.coef_)
attention = raw_weights / raw_weights.sum()  # softmax-like

print('Attention weights by lag (most recent = lag 1):')
for lag in range(1, 6):
    print(f'  Lag {lag:2d}: {attention[-lag]:.4f} ({"high" if attention[-lag] > attention.mean() else "low"})')
print(f'Top-3 most attended lags: {(-attention).argsort()[:3] + 1}')

**N-BEATS-inspired basis expansion forecasting**

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge

np.random.seed(0)
n = 300
t = np.arange(n)
y = 10*np.sin(0.2*t) + 0.1*t + np.random.normal(0, 1.5, n)

# N-BEATS idea: decompose into trend + seasonality bases
def trend_basis(T, degree=3):
    """Polynomial trend basis for N-BEATS trend block."""
    t_norm = np.linspace(0, 1, T)
    return np.column_stack([t_norm**k for k in range(degree+1)])

def fourier_basis(T, harmonics=5):
    """Fourier seasonality basis for N-BEATS seasonality block."""
    t = np.linspace(0, 2*np.pi, T)
    cols = []
    for k in range(1, harmonics+1):
        cols.extend([np.cos(k*t), np.sin(k*t)])
    return np.column_stack(cols)

LB = 30  # lookback
X, Y = [], []
for i in range(LB, n - 7):
    window = y[i-LB:i]
    trend_feats  = trend_basis(LB, degree=2).T @ window / LB
    season_feats = fourier_basis(LB, harmonics=3).T @ window / LB
    X.append(np.concatenate([trend_feats, season_feats]))
    Y.append(y[i:i+7].mean())
X, Y = np.array(X), np.array(Y)

split = int(0.8*len(X))
model = Ridge(alpha=0.1).fit(X[:split], Y[:split])
pred  = model.predict(X[split:])
rmse  = np.sqrt(((Y[split:] - pred)**2).mean())
print(f'N-BEATS basis expansion RMSE: {rmse:.3f}')
print(f'Trend features: {trend_basis(LB, 2).shape[1]}, Fourier features: {fourier_basis(LB, 3).shape[1]}')

### Real-World Scenario

> Build a multi-step electricity demand forecasting system that predicts next 24 hours (hourly) using the previous 7 days of data. Features include lagged values, hour-of-day, day-of-week, temperature. Compare gradient boosting seq2seq vs naive persistence baseline.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import GradientBoostingRegressor

np.random.seed(5)
hours = pd.date_range('2023-01-01', periods=8760, freq='H')
daily = 500 + 200*np.sin(2*np.pi*np.arange(8760)/24 - np.pi)
weekly= 50 *np.sin(2*np.pi*np.arange(8760)/(24*7))
noise = np.random.normal(0, 20, 8760)
demand = daily + weekly + noise

df = pd.DataFrame({'time': hours, 'demand': demand, 'hour': hours.hour, 'dow': hours.dayofweek})

LB, H = 24*7, 24  # 7-day lookback, 24h horizon
X, Y = [], []
for i in range(LB, len(df)-H):
    feats = list(df['demand'].values[i-LB:i])
    feats += [df['hour'].iloc[i], df['dow'].iloc[i]]
    X.append(feats)
    Y.append(df['demand'].values[i:i+H])
X, Y = np.array(X), np.array(Y)

split = int(0.85*len(X))
model = MultiOutputRegressor(GradientBoostingRegressor(n_estimators=30, random_state=0), n_jobs=-1)
model.fit(X[:split], Y[:split])
pred = model.predict(X[split:])
rmse = np.sqrt(((Y[split:]-pred)**2).mean())
baseline = np.sqrt(((Y[split:]-Y[split-1:-1])**2).mean())  # persistence
print(f'GBM RMSE: {rmse:.2f}, Persistence RMSE: {baseline:.2f}')
print(f'Improvement: {(baseline-rmse)/baseline:.1%}')

### Practice: Seq2Seq Solar Power Forecasting

Using 1 year of hourly solar generation data (sinusoidal with daily pattern, zero at night), build a seq2seq GBM model with 48-hour lookback and 12-hour horizon. Compare RMSE for the first 4 steps vs last 4 steps. Also implement a simple attention weighting by computing per-lag correlations with the target.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import GradientBoostingRegressor

np.random.seed(3)
hours = pd.date_range('2023-01-01', periods=8760, freq='H')
raw = np.maximum(0, np.sin(2*np.pi*np.arange(8760)/24 - np.pi/2))
solar = raw * 1000 + np.random.normal(0, 30, 8760)
df = pd.DataFrame({'time': hours, 'solar': solar})

LB, H = 48, 12
# TODO: Build (X, Y) windows with lookback=48, horizon=12
# TODO: Train GBM MultiOutputRegressor
# TODO: Report RMSE for steps 1-4 vs steps 9-12
# TODO: Compute attention weights via per-lag correlation


## 14. 14. Wavelet Analysis for Time Series



**Discrete Wavelet Transform (DWT)**

In [ ]:
import numpy as np
import pywt
np.random.seed(0)
t = np.linspace(0, 8*np.pi, 512)
signal = (np.sin(t) + 0.5*np.sin(5*t) + np.random.normal(0, 0.2, 512))
wavelet = 'db4'
level = 4
coeffs = pywt.wavedec(signal, wavelet, level=level)
print(f"Decomposition levels: {len(coeffs)-1}")
print(f"Approximation coeff shape: {coeffs[0].shape}")
for i, c in enumerate(coeffs[1:], 1):
    print(f"  Detail level {i}: shape={c.shape}, energy={np.sum(c**2):.2f}")
# Denoise: zero out small detail coefficients
threshold = 0.3
coeffs_denoised = [coeffs[0]] + [pywt.threshold(c, threshold, mode='soft') for c in coeffs[1:]]
denoised = pywt.waverec(coeffs_denoised, wavelet)[:len(signal)]
mse = np.mean((signal - denoised)**2)
print(f"Denoising MSE: {mse:.4f}")

**Continuous Wavelet Transform (CWT) Scalogram**

In [ ]:
import numpy as np
import pywt
np.random.seed(1)
fs = 100  # Hz
t = np.arange(0, 4, 1/fs)
# Chirp: frequency increases from 5 to 20 Hz
freq = 5 + 15*t/4
signal = np.sin(2*np.pi*freq*t) + np.random.normal(0, 0.1, len(t))
widths = np.arange(1, 64)
cwt_matrix, freqs = pywt.cwt(signal, widths, 'morl', sampling_period=1/fs)
power = np.abs(cwt_matrix)**2
# Peak frequency at start vs end
t_start = slice(0, 50); t_end = slice(350, 400)
peak_freq_start = freqs[np.argmax(power[:, t_start].mean(axis=1))]
peak_freq_end   = freqs[np.argmax(power[:, t_end].mean(axis=1))]
print(f"Dominant frequency (start): {peak_freq_start:.1f} Hz")
print(f"Dominant frequency (end):   {peak_freq_end:.1f} Hz")

**Wavelet-Based Anomaly Detection**

In [ ]:
import numpy as np
import pywt
np.random.seed(42)
n = 1000
ts = np.sin(2*np.pi*np.arange(n)/50) + np.random.normal(0, 0.1, n)
# Inject anomalies
ts[300:310] += 3.0
ts[700] -= 4.0
# DWT anomaly detection via reconstruction error per window
coeffs = pywt.wavedec(ts, 'haar', level=3)
# Zero out detail at level 1 (high-freq noise)
coeffs[1] = np.zeros_like(coeffs[1])
reconstructed = pywt.waverec(coeffs, 'haar')[:n]
residuals = np.abs(ts - reconstructed)
threshold = residuals.mean() + 3*residuals.std()
anomaly_idx = np.where(residuals > threshold)[0]
print(f"Anomalies detected at indices: {anomaly_idx[:10]}")
print(f"True anomaly regions: 300-310, 700")

### Real-World Scenario

> Manufacturing: use wavelet decomposition to separate machine vibration signals by frequency band, detect bearing faults (high-frequency detail) while ignoring normal operational frequencies.

In [ ]:
import numpy as np
import pywt
np.random.seed(7)
fs = 1000  # 1 kHz sampling
t = np.arange(0, 2, 1/fs)  # 2 seconds
# Normal operation: 10 Hz fundamental + harmonics
normal = (np.sin(2*np.pi*10*t) + 0.3*np.sin(2*np.pi*20*t) +
          np.random.normal(0, 0.05, len(t)))
# Fault: adds 150 Hz bearing frequency
fault  = normal + 0.5*np.sin(2*np.pi*150*t)
for label, sig in [('Normal', normal), ('Fault', fault)]:
    coeffs = pywt.wavedec(sig, 'db4', level=5)
    # Level 1 detail captures highest frequencies
    hf_energy = np.sum(coeffs[1]**2) / np.sum(sig**2)
    print(f"{label}: high-freq energy ratio = {hf_energy:.4f}")
    energies = [np.sum(c**2) / np.sum(sig**2) for c in coeffs]
    print(f"  Energy by level: " + ", ".join(f"L{i}={e:.3f}" for i, e in enumerate(energies)))

### Practice: EEG Brainwave Analysis

Given a simulated EEG signal (1-second, 256 Hz) with delta (1-4 Hz), alpha (8-13 Hz), and beta (13-30 Hz) components, use wavelet decomposition to extract each band. Report the relative power of each band. Inject a 50ms epileptic spike at t=0.5s and detect it using wavelet residuals.

In [ ]:
import numpy as np
import pywt
fs = 256  # Hz
t = np.arange(0, 1, 1/fs)
# Multi-band EEG signal
delta = 0.5 * np.sin(2*np.pi*2*t)
alpha = 0.3 * np.sin(2*np.pi*10*t)
beta  = 0.2 * np.sin(2*np.pi*20*t)
noise = np.random.normal(0, 0.05, len(t))
eeg = delta + alpha + beta + noise
# Inject spike
spike_start = int(0.5 * fs)
eeg[spike_start:spike_start+13] += 3.0
# TODO: DWT decomposition with 'db4', level=5
# TODO: Identify which detail levels correspond to each EEG band
# TODO: Compute relative band powers
# TODO: Detect spike using reconstruction error


## 15. 15. Multivariate Time Series & VAR Models



**Vector Autoregression (VAR)**

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.vector_ar.var_model import VAR
np.random.seed(42)
n = 200
e1, e2 = np.random.normal(0, 1, n), np.random.normal(0, 1, n)
y1, y2 = np.zeros(n), np.zeros(n)
for t in range(1, n):
    y1[t] = 0.6*y1[t-1] + 0.2*y2[t-1] + e1[t]
    y2[t] = 0.1*y1[t-1] + 0.7*y2[t-1] + e2[t]
df = pd.DataFrame({'y1': y1, 'y2': y2})
model = VAR(df)
results = model.fit(maxlags=4, ic='aic')
print(f"Selected lag order: {results.k_ar}")
forecast = results.forecast(df.values[-results.k_ar:], steps=5)
print("5-step forecast:"); print(forecast.round(3))

**Granger Causality Test**

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
np.random.seed(1)
n = 300
eps1 = np.random.normal(0, 1, n)
eps2 = np.random.normal(0, 1, n)
x = np.zeros(n); y = np.zeros(n)
for t in range(2, n):
    x[t] = 0.5*x[t-1] + eps1[t]
    y[t] = 0.4*x[t-1] + 0.3*y[t-1] + eps2[t]  # x Granger-causes y
df = pd.DataFrame({'y': y, 'x': x})
max_lag = 4
print("Granger causality (x -> y):")
res = grangercausalitytests(df[['y','x']], maxlag=max_lag, verbose=False)
for lag, r in res.items():
    f_stat = r[0]['ssr_ftest'][0]
    p_val  = r[0]['ssr_ftest'][1]
    print(f"  Lag {lag}: F={f_stat:.3f}, p={p_val:.4f}")

**Cointegration & Error Correction**

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import coint
from statsmodels.regression.linear_model import OLS
np.random.seed(5)
n = 500
common_trend = np.cumsum(np.random.normal(0, 1, n))
y1 = common_trend + np.random.normal(0, 0.5, n)
y2 = 0.8 * common_trend + 1.2 + np.random.normal(0, 0.5, n)
score, pvalue, _ = coint(y1, y2)
print(f"Cointegration test: t={score:.4f}, p={pvalue:.4f}")
# Estimate cointegrating vector
beta = OLS(y1, np.column_stack([np.ones(n), y2])).fit().params
spread = y1 - beta[1]*y2 - beta[0]
z_spread = (spread - spread.mean()) / spread.std()
print(f"Spread mean-reversion: current z-score = {z_spread[-1]:.3f}")

### Real-World Scenario

> Macroeconomic forecasting: model GDP, inflation, and interest rates jointly using VAR to capture cross-variable dynamics and produce scenario forecasts for monetary policy analysis.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.stattools import grangercausalitytests
np.random.seed(10)
n = 120  # 10 years monthly
# Simulate: interest rate affects inflation with lag, both affect GDP
ir = np.cumsum(np.random.normal(0, 0.1, n)) + 3.0
inf = 0.5*np.roll(ir, 2) + np.cumsum(np.random.normal(0, 0.05, n)) + 2.0
gdp = -0.3*np.roll(ir, 3) + 0.4*np.roll(inf, 1) + np.cumsum(np.random.normal(0, 0.08, n)) + 2.5
df = pd.DataFrame({'gdp': gdp, 'inflation': inf, 'interest_rate': ir})
df = df.diff().dropna()  # difference to achieve stationarity
model = VAR(df)
res = model.fit(maxlags=6, ic='aic')
print(f"VAR lag order: {res.k_ar}")
forecast = res.forecast(df.values[-res.k_ar:], steps=12)
print(f"12-month GDP forecast range: {forecast[:,0].min():.3f} to {forecast[:,0].max():.3f}")
# Impulse response
irf = res.irf(10)
print(f"GDP response to interest shock at lag 3: {irf.orth_irfs[3, 0, 2]:.4f}")

### Practice: Pairs Trading Signal

Using two simulated stock price series that are cointegrated (sharing a common random walk), implement a pairs trading strategy: (1) verify cointegration with Engle-Granger test, (2) estimate the spread, (3) generate long/short signals when z-score > 2 or < -2, (4) backtest and report Sharpe ratio and max drawdown.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import coint
from statsmodels.regression.linear_model import OLS
np.random.seed(3)
n = 500
common = np.cumsum(np.random.normal(0, 1, n))
price_a = common + np.random.normal(0, 0.5, n) + 50
price_b = 0.9*common + np.random.normal(0, 0.5, n) + 45
# TODO: Cointegration test
# TODO: Compute hedge ratio and spread
# TODO: Z-score of spread
# TODO: Generate trading signals (long A-short B when z<-2, vice versa)
# TODO: Compute daily P&L, Sharpe ratio, max drawdown


## 16. 16. Real-Time Streaming & Online Learning



**Online Learning with River**

In [ ]:
# pip install river
from river import linear_model, preprocessing, metrics, stream
import numpy as np
np.random.seed(0)
n = 1000
X_data = np.column_stack([np.random.randn(n), np.arange(n)/n])
y_data = 3*X_data[:,0] - 2*X_data[:,1] + 0.5 + np.random.normal(0, 0.3, n)
model = preprocessing.StandardScaler() | linear_model.LinearRegression()
metric = metrics.RMSE()
for i, (xi, yi) in enumerate(zip(X_data, y_data)):
    x_dict = {'f0': xi[0], 'f1': xi[1]}
    y_pred = model.predict_one(x_dict)
    if y_pred is not None:
        metric.update(yi, y_pred)
    model.learn_one(x_dict, yi)
    if i % 200 == 0 and y_pred is not None:
        print(f"  n={i}: running RMSE={metric.get():.4f}")
print(f"Final RMSE: {metric.get():.4f}")

**Concept Drift Detection (ADWIN)**

In [ ]:
# pip install river
from river import drift
import numpy as np
np.random.seed(7)
detector = drift.ADWIN()
stream = np.concatenate([
    np.random.normal(0, 1, 500),
    np.random.normal(3, 1, 500)  # sudden drift at t=500
])
drifts = []
for i, x in enumerate(stream):
    detector.update(x)
    if detector.drift_detected:
        drifts.append(i)
        detector = drift.ADWIN()  # reset
print(f"Drift detected at sample(s): {drifts}")

**Sliding Window Streaming Statistics**

In [ ]:
import numpy as np
from collections import deque
class StreamStats:
    def __init__(self, window=100):
        self.w = deque(maxlen=window)
        self.n = 0
    def update(self, x):
        self.w.append(x)
        self.n += 1
        return self.mean(), self.std()
    def mean(self): return np.mean(self.w)
    def std(self):  return np.std(self.w, ddof=1) if len(self.w) > 1 else 0.0
np.random.seed(1)
ss = StreamStats(window=50)
for i in range(300):
    val = np.random.normal(5 if i < 150 else 8, 1)
    mean, std = ss.update(val)
    if i % 50 == 49:
        print(f"  t={i+1}: window mean={mean:.3f}, std={std:.3f}")

### Real-World Scenario

> IoT sensor network: process a continuous stream of temperature readings at 10 Hz from 50 sensors, detect anomalies in real-time using sliding window statistics, and adapt the baseline when ADWIN detects environmental regime changes.

In [ ]:
import numpy as np
from collections import deque
np.random.seed(42)
n_sensors, n_steps = 5, 1000
window = 60
baselines = [deque(maxlen=window) for _ in range(n_sensors)]
alerts = {i: [] for i in range(n_sensors)}
# Inject drift at step 600 for sensor 2
for t in range(n_steps):
    readings = np.random.normal(22, 0.5, n_sensors)
    if t >= 600:
        readings[2] += 4.0  # sensor 2 drifts
    for s in range(n_sensors):
        baselines[s].append(readings[s])
        if len(baselines[s]) >= 20:
            mu = np.mean(baselines[s])
            sigma = np.std(baselines[s], ddof=1)
            z = (readings[s] - mu) / max(sigma, 1e-6)
            if abs(z) > 3:
                alerts[s].append(t)
for s in range(n_sensors):
    if alerts[s]:
        print(f"Sensor {s}: {len(alerts[s])} alerts, first at t={alerts[s][0]}")
    else:
        print(f"Sensor {s}: no alerts")

### Practice: Real-Time Fraud Score Streaming

Simulate a stream of 2000 credit card transactions (amount, time_since_last, is_weekend). Use online logistic regression (River) trained incrementally. At t=1000, inject a concept drift (fraudsters change behavior). Detect drift using ADWIN on prediction errors. Report AUC before and after drift, and adaptation speed.

In [ ]:
# pip install river
from river import linear_model, preprocessing, metrics, drift, stream as rv_stream
import numpy as np
np.random.seed(99)
n = 2000
# Generate transactions
amounts = np.random.exponential(50, n)
gaps = np.random.exponential(10, n)
is_weekend = np.random.randint(0, 2, n)
# Fraud rule: changes at t=1000
fraud_prob_before = 1 / (1 + np.exp(-(amounts/100 - 0.5)))
fraud_prob_after  = 1 / (1 + np.exp(-(gaps/5 - 1.0)))  # different pattern
y = np.array([
    np.random.binomial(1, fraud_prob_before[i]) if i < 1000
    else np.random.binomial(1, fraud_prob_after[i])
    for i in range(n)
])
# TODO: Online logistic regression with River
# TODO: ADWIN drift detection on binary prediction errors
# TODO: Report AUC in windows before drift, at drift, after adaptation
